# Quais estatísticas têm maior peso no número de vitórias da temporada?
# Quais os valores dessas estatísticas para as vitórias necessárias para o primeiro lugar na temporada 2026/2027?

In [1]:
# Biblioteca de dataframe
import pandas as pd
import numpy as np

# Biblioteca de visualização de dados
import seaborn as sns
import matplotlib  as plt
import plotly.express as px
import plotly.graph_objects as go

# Biblioteca de preparação do modelo
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score
from sklearn.ensemble import StackingRegressor

# Biblioteca de modelagem
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Bibliotecas de padronização
from sklearn.preprocessing import StandardScaler

In [2]:
# Importar as tabelas das últimas temporadas
df_16 = pd.read_csv(r'Classificacao_NBB\NBB_16_teams.csv')
df_17 = pd.read_csv(r'Classificacao_NBB\NBB_17_teams.csv')
df_18 = pd.read_csv(r'Classificacao_NBB\NBB_18_teams.csv')
df_20 = pd.read_csv(r'Classificacao_NBB\NBB_20_teams.csv')
df_21 = pd.read_csv(r'Classificacao_NBB\NBB_21_teams.csv')
df_22 = pd.read_csv(r'Classificacao_NBB\NBB_22_teams.csv')
df_23 = pd.read_csv(r'Classificacao_NBB\NBB_23_teams.csv')
df_24 = pd.read_csv(r'Classificacao_NBB\NBB_24_teams.csv')
df_25 = pd.read_csv(r'Classificacao_NBB\NBB_25_teams.csv')

In [3]:
# Conferir se o dataframe está apresentado corretamente
df_21.head()

,Team,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,...,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
0,Franca,88.0,31.3,65.4,0.478,10.7,30.0,0.359,14.7,19.0,...,13.5,7.6,7.6,118.7,103.5,15.2,74.0,1.5,38,0.884
1,Flamengo,85.9,30.1,65.8,0.458,12.6,34.6,0.363,13.1,17.8,...,14.7,8.7,6.0,118.0,100.7,17.3,72.7,1.5,33,0.786
2,Sao Paulo FC,82.8,29.8,61.7,0.482,10.0,28.4,0.351,13.3,18.3,...,14.3,9.5,7.1,113.4,102.2,11.2,72.1,1.5,27,0.692
3,Minas,82.8,29.5,62.3,0.474,10.0,28.1,0.356,13.7,18.3,...,15.6,9.6,6.7,114.7,106.4,8.4,72.0,1.3,28,0.718
4,Pinheiros Sky,80.9,29.4,69.8,0.422,9.6,28.9,0.334,12.4,17.4,...,14.8,10.1,7.9,107.4,108.1,-0.6,74.9,1.3,17,0.459


In [4]:
# Vendo lista de colunas presentes nos dataframes para facilitar análises posteriormente
df_21.columns

Index(['Team', 'PPG', 'FGM', 'FGA', 'FG%', '3PM', '3PA', '3P%', 'FTM', 'FTA',
       'FT%', 'ORB', 'DRB', 'RPG', 'APG', 'SPG', 'BPG', 'TOV', 'TS%', 'eFG%',
       'ORB%', 'DRB%', 'TRB%', 'AST%', 'TOV%', 'STL%', 'BLK%', 'ORtg', 'DRtg',
       'eDiff', 'Pace', 'Ast/TO', 'W's', 'Win %'],
      dtype='object')

## A estatística Pace tem relação com o número de vitórias na temporada?

In [5]:
# Gráfico de distribuição do Pace x W's
# Adiciona cada dataframe ao gráfico
fig = go.Figure(
    data=[
        go.Scatter(name = "21/22", x=df_21["Pace"], y=df_21["W's"], mode='markers', text=df_21["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "22/23", x=df_22["Pace"], y=df_22["W's"], mode='markers', text=df_22["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "23/24", x=df_23["Pace"], y=df_23["W's"], mode='markers', text=df_23["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "24/25", x=df_24["Pace"], y=df_24["W's"], mode='markers', text=df_24["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "25/26", x=df_25["Pace"], y=df_25["W's"], mode='markers', text=df_25["Team"], marker=dict(color='#A1A2A5'), showlegend=False)
    ])

# Adiciona a logo dos times que estão sendo analisados ao gráfico
logos_times = ['Franca', 'Minas', 'Bauru', 'Flamengo', 'Uniceub-BRB-Brasilia']
for df in [df_21, df_22, df_23, df_24, df_25]:
    for i, row in df.iterrows():
        if row['Team'] in logos_times:
            fig.add_layout_image(
                dict(
                    source = f"Logos/{row['Team']}.png",
                    x = row["Pace"],
                    y = row["W's"],
                    xref="x",
                    yref="y",
                    sizex=4,
                    sizey=4,
                    xanchor="center",
                    yanchor="middle",
                    layer="above"
                )
            )
# Adiciona a média de pace para comparação
pace_avg = (df_21["Pace"].mean() +
            df_22["Pace"].mean() +
            df_23["Pace"].mean() +
            df_24["Pace"].mean() +
            df_25["Pace"].mean())/ 5

df_total = pd.concat([df_16, df_17, df_18, df_20, df_21, df_22, df_23, df_24, df_25], ignore_index=True)

fig.add_trace(
    go.Scatter(
        x=[pace_avg, pace_avg],
        y=[df_total["W's"].min(), df_total["W's"].max()],
        mode="lines",
        line=dict(color="#1C2846", dash="dash"),
        name="Média de Pace"
    )
)

# Faz ajustes na apresentação do gráfico
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            text="Pace",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            text="Vitórias",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    title =dict(
        text="<b>Pace x Vitórias<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )

)
fig.show()
#fig.write_image("PacexVitorias.png")

É possível perceber que não há correlação clara entre o ritmo de jogo médio de um time (Pace) e sua quantidade de vitórias na temporada.

Porém podemos notar que existe uma concentração dos times com maior número de vitórias próximo à média de Pace da liga.

## Existe correlação entre número de vitórias e volume de tentativas de 3pts por tentativas totais?

In [6]:
# Gráfico de distribuição do Vol3pt x W's
# Adiciona cada dataframe ao gráfico
fig = go.Figure(
    data=[
        go.Scatter(name = "21/22", x=(df_21["3PA"]/df_21["FGA"])*100, y=df_21["W's"], mode='markers', text=df_21["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "22/23", x=(df_22["3PA"]/df_22["FGA"])*100, y=df_22["W's"], mode='markers', text=df_22["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "23/24", x=(df_23["3PA"]/df_23["FGA"])*100, y=df_23["W's"], mode='markers', text=df_23["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "24/25", x=(df_24["3PA"]/df_24["FGA"])*100, y=df_24["W's"], mode='markers', text=df_24["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "25/26", x=(df_25["3PA"]/df_25["FGA"])*100, y=df_25["W's"], mode='markers', text=df_25["Team"], marker=dict(color='#A1A2A5'), showlegend=False)
    ])

# Adiciona a logo dos times que estão sendo analisados ao gráfico
logos_times = ['Franca', 'Minas', 'Bauru', 'Flamengo', 'Uniceub-BRB-Brasilia']
for df in [df_21, df_22, df_23, df_24, df_25]:
    for i, row in df.iterrows():
        if row['Team'] in logos_times:
            fig.add_layout_image(
                dict(
                    source = f"Logos/{row['Team']}.png",
                    x = (row["3PA"]/row["FGA"])*100,
                    y = row["W's"],
                    xref="x",
                    yref="y",
                    sizex=4,
                    sizey=4,
                    xanchor="center",
                    yanchor="middle",
                    layer="above"
                )
            )
# Adiciona a média de volume de 3pt para comparação
vol_avg = (((df_21["3PA"]/df_21["FGA"])*100).mean() +
           ((df_22["3PA"]/df_22["FGA"])*100).mean() +
           ((df_23["3PA"]/df_23["FGA"])*100).mean() +
           ((df_24["3PA"]/df_24["FGA"])*100).mean() +
           ((df_25["3PA"]/df_25["FGA"])*100).mean())/ 5

fig.add_trace(
    go.Scatter(
        x=[vol_avg, vol_avg],
        y=[df_total["W's"].min(), df_total["W's"].max()],
        mode="lines",
        line=dict(color="#1C2846", dash="dash"),
        name="Média de volume de 3PT"
    )
)

# Faz ajustes na apresentação do gráfico
fig.update_layout(
    template="simple_white",
    xaxis=dict(
        title=dict(
            text="Volume de 3pt (%)",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            text="Vitórias",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    title =dict(
        text="<b>Volume de 3pt x Vitórias<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )

)
fig.show()
#fig.write_image("Vol3ptxVitorias.png")

Não há uma correlação clara entre o volume de tentativas de 3PT e a quantidade de vitórias, mas uma boa parte dos times com maior número de vitórias se encontram perto da média da liga (entre 43.5% e 45.8%)

## Existe uma relação entre o percentual de rebotes totais e o netrating do time?

In [7]:
# Gráfico de distribuição do Total Reb x NetRating
# Adiciona cada dataframe ao gráfico
fig = go.Figure(
    data=[
        go.Scatter(name = "21/22", x=(df_21["TRB%"]), y=df_21["eDiff"], mode='markers', text=df_21["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "22/23", x=(df_22["TRB%"]), y=df_22["eDiff"], mode='markers', text=df_22["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "23/24", x=(df_23["TRB%"]), y=df_23["eDiff"], mode='markers', text=df_23["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "24/25", x=(df_24["TRB%"]), y=df_24["eDiff"], mode='markers', text=df_24["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "25/26", x=(df_25["TRB%"]), y=df_25["eDiff"], mode='markers', text=df_25["Team"], marker=dict(color='#A1A2A5'), showlegend=False)
    ])

# Adiciona a logo dos times que estão sendo analisados ao gráfico
logos_times = ['Franca', 'Minas', 'Bauru', 'Flamengo', 'Uniceub-BRB-Brasilia']
for df in [df_21, df_22, df_23, df_24, df_25]:
    for i, row in df.iterrows():
        if row['Team'] in logos_times:
            fig.add_layout_image(
                dict(
                    source = f"Logos/{row['Team']}.png",
                    x = row["TRB%"],
                    y = row["eDiff"],
                    xref="x",
                    yref="y",
                    sizex=4,
                    sizey=4,
                    xanchor="center",
                    yanchor="middle",
                    layer="above"
                )
            )
# Adiciona a média de Rebotes totais para comparação
trb_avg = ((df_21["TRB%"]).mean() +
           (df_22["TRB%"]).mean() +
           (df_23["TRB%"]).mean() +
           (df_24["TRB%"]).mean() +
           (df_25["TRB%"]).mean())/ 5

fig.add_trace(
    go.Scatter(
        x=[trb_avg, trb_avg],
        y=[df_total["eDiff"].min(), df_total["eDiff"].max()],
        mode="lines",
        line=dict(color="#1C2846", dash="dash"),
        name="Média de total de rebotes (%)"
    )
)

# Faz ajustes na apresentação do gráfico
fig.update_layout(
    template="simple_white",
    xaxis=dict(
        title=dict(
            text="Total de rebotes (%)",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            text="Net Rating",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    title =dict(
        text="<b>Total de rebotes (%) x Net Rating<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )

)
fig.show()
#fig.write_image("TRB%xNetRating.png")

A análise retornou uma correlação positiva. Logo, os times que pegam mais rebotes de todas as tentativas (ofensivas e defensivas) tendem a ter uma performance líquida melhor.

É importante perceber que o time do Brasília Basquete se apresentou como outlier em duas situações:

A primeira na temporada 2023/2024, na qual teve um total percentual de rebotes baixo, e, por isso, obteve uma performance líquida baixa;

A segunda na temporada 2021/2022, na qual, mesmo tendo um percentual de rebotes acima da média geral da liga, obteve uma performance líquida negativa.

## Existe correlação entre o percentual de assistências por acertos e a porcentagem efetiva de acertos?

In [8]:
# Gráfico de distribuição do percentual de assistências x percentual efetivo de acertos
# Adiciona cada dataframe ao gráfico
fig = go.Figure(
    data=[
        go.Scatter(name = "21/22", x=(df_21["AST%"]), y=(df_21["eFG%"])*100, mode='markers', text=df_21["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "22/23", x=(df_22["AST%"]), y=(df_22["eFG%"])*100, mode='markers', text=df_22["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "23/24", x=(df_23["AST%"]), y=(df_23["eFG%"])*100, mode='markers', text=df_23["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "24/25", x=(df_24["AST%"]), y=(df_24["eFG%"])*100, mode='markers', text=df_24["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "25/26", x=(df_25["AST%"]), y=(df_25["eFG%"])*100, mode='markers', text=df_25["Team"], marker=dict(color='#A1A2A5'), showlegend=False)
    ])

# Adiciona a logo dos times que estão sendo analisados ao gráfico
logos_times = ['Franca', 'Minas', 'Bauru', 'Flamengo', 'Uniceub-BRB-Brasilia']
for df in [df_21, df_22, df_23, df_24, df_25]:
    for i, row in df.iterrows():
        if row['Team'] in logos_times:
            fig.add_layout_image(
                dict(
                    source = f"Logos/{row['Team']}.png",
                    x = row["AST%"],
                    y = (row["eFG%"])*100,
                    xref="x",
                    yref="y",
                    sizex=2,
                    sizey=2,
                    xanchor="center",
                    yanchor="middle",
                    layer="above"
                )
            )
# Adiciona a média de Rebotes totais para comparação
ast_avg = ((df_21["AST%"]).mean() +
           (df_22["AST%"]).mean() +
           (df_23["AST%"]).mean() +
           (df_24["AST%"]).mean() +
           (df_25["AST%"]).mean())/ 5

fig.add_trace(
    go.Scatter(
        x=[ast_avg, ast_avg],
        y=[40, 60],
        mode="lines",
        line=dict(color="#1C2846", dash="dash"),
        name="Média de assistências (%)"
    )
)

# Faz ajustes na apresentação do gráfico
fig.update_layout(
    template="simple_white",
    xaxis=dict(
        title=dict(
            text="Assistências (%)",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    yaxis=dict(
        range=[40, 60],
        title=dict(
            text="Acertos efetivos (%)",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    title =dict(
        text="<b>Assistências (%) x Acertos efetivos (%)<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )

)
fig.show()
#fig.write_image("AST%xeFG%.png")

A análise demonstra que não há correlação entre o percentual de assistências em tentativas certas, e o percentual de acertos efetivos.

Isso indica que ter uma boa parte das jogadas convertidas terminando em passes não garante um percentual de conversões efetivas maior.

## Existe correlação entre o Net Rating e o percentual de vitórias na temporada?

In [9]:
# Gráfico de distribuição do NetRating x Win %
# Adiciona cada dataframe ao gráfico
fig = go.Figure(
    data=[
        go.Scatter(name = "21/22", x=(df_21["eDiff"]), y=(df_21["Win %"])*100, mode='markers', text=df_21["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "22/23", x=(df_22["eDiff"]), y=(df_22["Win %"])*100, mode='markers', text=df_22["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "23/24", x=(df_23["eDiff"]), y=(df_23["Win %"])*100, mode='markers', text=df_23["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "24/25", x=(df_24["eDiff"]), y=(df_24["Win %"])*100, mode='markers', text=df_24["Team"], marker=dict(color='#A1A2A5'), showlegend=False),
        go.Scatter(name = "25/26", x=(df_25["eDiff"]), y=(df_25["Win %"])*100, mode='markers', text=df_25["Team"], marker=dict(color='#A1A2A5'), showlegend=False)
    ])

# Adiciona a logo dos times que estão sendo analisados ao gráfico
logos_times = ['Franca', 'Minas', 'Bauru', 'Flamengo', 'Uniceub-BRB-Brasilia']
for df in [df_21, df_22, df_23, df_24, df_25]:
    for i, row in df.iterrows():
        if row['Team'] in logos_times:
            fig.add_layout_image(
                dict(
                    source = f"Logos/{row['Team']}.png",
                    x = row["eDiff"],
                    y = (row["Win %"])*100,
                    xref="x",
                    yref="y",
                    sizex=8,
                    sizey=8,
                    xanchor="center",
                    yanchor="middle",
                    layer="above"
                )
            )
# Adiciona a média de Rebotes totais para comparação
net_avg = ((df_21["eDiff"]).mean() +
           (df_22["eDiff"]).mean() +
           (df_23["eDiff"]).mean() +
           (df_24["eDiff"]).mean() +
           (df_25["eDiff"]).mean())/ 5

fig.add_trace(
    go.Scatter(
        x=[net_avg, net_avg],
        y=[(df_total["Win %"].min())*100, (df_total["Win %"].max())*100],
        mode="lines",
        line=dict(color="#1C2846", dash="dash"),
        name="Média de Net Rating"
    )
)

# Faz ajustes na apresentação do gráfico
fig.update_layout(
    template="simple_white",
    xaxis=dict(
        title=dict(
            text="Net Rating",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            text="Vitórias (%)",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        showgrid=False,
        showline=True,
        linecolor="black",
    ),
    title =dict(
        text="<b>Net Rating x Vitórias (%)<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )

)
fig.show()
#fig.write_image("NetRatingxW%.png")

A análise demonstra haver correlação positiva entre a performance líquida de um time, e o percentual de vitórias daquele time na temporada.

A análise também demonstra que o time do Brasília Basquete conseguiu obter mais que 50% de vitórias quando superou a média de performance líquida da liga na temporada 2024/2025

# Gráficos de correlação entre estatísticas

## Correlação de estatísticas para a temporada 2021/2022

In [10]:
# Criar a variável que calcula as correlações entre valores das colunas
corr_21 = df_21.select_dtypes(include = ['number']).corr()
color_scale = [[0,"#A1A2A5"], [1,"#1C2846"]]
# Criar o gráfico para visualização do resultado e insights
fig = px.imshow(
    corr_21,
    text_auto = True,
    aspect = 'auto',
    color_continuous_scale=color_scale
)
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text="<b>Correlação de variáveis 2021/2022<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

Com relação a tabela da temporada 2021/2022 temos alguns dados interessantes de correlação com o número de vitórias.

Os que demonstram correlação positiva, excluído a variável alvo (W%), são:

* ASS/TO = 0,86
* eDiff = 0,95
* ORTG = 0,93
* TS% = 0,90
* eFG% = 0,91
* FT% = 0,70
* 3PT% = 0,83
* FG% = 0,87
* PPG = 0,87

Os que demonstram correlação negativa são:

* DRTG = -0,81
* TOV = -0,61

## Correlação de estatísticas para a temporada 2022/2023

In [11]:
# Criar a variável que calcula as correlações entre valores das colunas
corr_22 = df_22.select_dtypes(include = ['number']).corr()

# Criar o gráfico para visualização do resultado e insights
fig = px.imshow(
    corr_22,
    text_auto = True,
    aspect = 'auto',
    color_continuous_scale=color_scale
)
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text="<b>Correlação de variáveis 2022/2023<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

As colunas correlacionadas se mantiveram, mas os valores apresentaram queda de ate 10% para essa temporada

## Correlação de estatísticas para a temporada 2023/2024

In [12]:
# Criar a variável que calcula as correlações entre valores das colunas
corr_23 = df_23.select_dtypes(include = ['number']).corr()

# Criar o gráfico para visualização do resultado e insights
fig = px.imshow(
    corr_23,
    text_auto = True,
    aspect = 'auto',
    color_continuous_scale=color_scale
)
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text="<b>Correlação de variáveis 2023/2024<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

Na temporada 2023/2024 o TOV% não teve forte correlação negativa.

Além disso 3PT% e FT% não apresentaram forte correlação positiva, ao invés disso algumas outras estatísticas apresentaram:

* APG = 0,81
* DRB = 0,75

## Correlação estatística da temporada 2024/2025

In [13]:
# Criar a variável que calcula as correlações entre valores das colunas
corr_24 = df_24.select_dtypes(include = ['number']).corr()

# Criar o gráfico para visualização do resultado e insights
fig = px.imshow(
    corr_24,
    text_auto = True,
    aspect = 'auto',
    color_continuous_scale=color_scale
)
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text="<b>Correlação de variáveis 2024/2025<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

Os resultados foram muito similares aos da correlação da temporada 2023/2024

## Correlação estatística da temporada 2025/2026

In [14]:
# Criar a variável que calcula as correlações entre valores das colunas
corr_24 = df_25.select_dtypes(include = ['number']).corr()

# Criar o gráfico para visualização do resultado e insights
fig = px.imshow(
    corr_24,
    text_auto = True,
    aspect = 'auto',
    color_continuous_scale=color_scale
)
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text="<b>Correlação de variáveis 2025/2026<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

Resultados muito similares às outras temporadas

# Criação de dataframe com os primeiros lugares

## O objetivo é ver quais as médias e desvios padrão para determinar um número realista de vitórias para o topo da tabela

In [15]:
# Criação do dataframe de primeiros lugares
df_1lug = pd.DataFrame()
for df in [df_16, df_17, df_18, df_20, df_21, df_22, df_23, df_24]:
 # pega a linha com maior valor de "W's"
    max_row = df.loc[df["W's"].idxmax()]
    # concatena no acumulador
    df_1lug = pd.concat([df_1lug, max_row.to_frame().T], ignore_index=True)

In [16]:
df_1lug

,Team,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,...,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
0,Bauru,81.5,27.9,61.6,0.453,9.8,28.0,0.348,16.0,22.1,...,15.7,9.7,5.3,110.5,102.5,8.0,72.8,1.3,30,0.667
1,Paulistano,84.3,29.1,66.9,0.435,12.3,34.0,0.363,13.8,18.2,...,13.9,9.1,7.1,115.0,99.8,15.2,73.1,1.6,32,0.8
2,Flamengo,83.6,29.9,66.3,0.451,9.2,27.3,0.338,14.5,19.0,...,14.4,9.9,4.4,114.2,97.2,17.0,73.3,1.5,31,0.816
3,Flamengo,89.2,30.7,68.1,0.451,13.3,37.1,0.359,14.4,18.8,...,13.0,9.3,4.4,119.7,97.6,22.1,74.4,1.8,36,0.947
4,Franca,88.0,31.3,65.4,0.478,10.7,30.0,0.359,14.7,19.0,...,13.5,7.6,7.6,118.7,103.5,15.2,74.0,1.5,38,0.884
5,Franca,90.8,31.4,64.2,0.49,11.1,28.9,0.385,16.8,21.5,...,14.2,7.2,6.9,121.2,107.0,14.2,74.1,1.5,41,0.872
6,Franca,85.7,29.9,62.5,0.479,10.3,27.4,0.376,15.6,19.6,...,15.7,8.4,7.3,117.6,102.7,15.0,72.4,1.5,42,0.84
7,Minas,81.1,29.7,65.4,0.453,9.1,26.6,0.341,12.8,18.4,...,14.9,9.8,8.5,111.6,98.0,13.5,72.8,1.3,39,0.83


In [17]:
# Identificação de outlier nas temporadas
fig = px.box(df_1lug, y="Win %")
fig.update_traces(
    marker_color="#1C2846"
)
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            text = "Porcentagem de vitórias",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text="<b>Boxplot de Win % nas temporadas<b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

In [18]:
# Tendo em vista que o melhor colocado da temporada 2016/2017 representa um outlier, eu decidi removê-lo do dataframe
df_1lug = df_1lug.drop(df.index[0])

df_1lug

,Team,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,...,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
1,Paulistano,84.3,29.1,66.9,0.435,12.3,34.0,0.363,13.8,18.2,...,13.9,9.1,7.1,115.0,99.8,15.2,73.1,1.6,32,0.8
2,Flamengo,83.6,29.9,66.3,0.451,9.2,27.3,0.338,14.5,19.0,...,14.4,9.9,4.4,114.2,97.2,17.0,73.3,1.5,31,0.816
3,Flamengo,89.2,30.7,68.1,0.451,13.3,37.1,0.359,14.4,18.8,...,13.0,9.3,4.4,119.7,97.6,22.1,74.4,1.8,36,0.947
4,Franca,88.0,31.3,65.4,0.478,10.7,30.0,0.359,14.7,19.0,...,13.5,7.6,7.6,118.7,103.5,15.2,74.0,1.5,38,0.884
5,Franca,90.8,31.4,64.2,0.49,11.1,28.9,0.385,16.8,21.5,...,14.2,7.2,6.9,121.2,107.0,14.2,74.1,1.5,41,0.872
6,Franca,85.7,29.9,62.5,0.479,10.3,27.4,0.376,15.6,19.6,...,15.7,8.4,7.3,117.6,102.7,15.0,72.4,1.5,42,0.84
7,Minas,81.1,29.7,65.4,0.453,9.1,26.6,0.341,12.8,18.4,...,14.9,9.8,8.5,111.6,98.0,13.5,72.8,1.3,39,0.83


In [19]:
# Vendo informações da tabela
df_1lug.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 1 to 7
Data columns (total 34 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Team    7 non-null      object
 1   PPG     7 non-null      object
 2   FGM     7 non-null      object
 3   FGA     7 non-null      object
 4   FG%     7 non-null      object
 5   3PM     7 non-null      object
 6   3PA     7 non-null      object
 7   3P%     7 non-null      object
 8   FTM     7 non-null      object
 9   FTA     7 non-null      object
 10  FT%     7 non-null      object
 11  ORB     7 non-null      object
 12  DRB     7 non-null      object
 13  RPG     7 non-null      object
 14  APG     7 non-null      object
 15  SPG     7 non-null      object
 16  BPG     7 non-null      object
 17  TOV     7 non-null      object
 18  TS%     7 non-null      object
 19  eFG%    7 non-null      object
 20  ORB%    7 non-null      object
 21  DRB%    7 non-null      object
 22  TRB%    7 non-null      object

In [20]:
# Convertendo tipos de valores
for column in df_1lug.columns:
    if column not in ["Team", "W's"]:
        df_1lug[column] = df_1lug[column].astype(float)

df_1lug["W's"] = df_1lug["W's"].astype(int)

In [21]:
df_1lug.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 1 to 7
Data columns (total 34 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Team    7 non-null      object 
 1   PPG     7 non-null      float64
 2   FGM     7 non-null      float64
 3   FGA     7 non-null      float64
 4   FG%     7 non-null      float64
 5   3PM     7 non-null      float64
 6   3PA     7 non-null      float64
 7   3P%     7 non-null      float64
 8   FTM     7 non-null      float64
 9   FTA     7 non-null      float64
 10  FT%     7 non-null      float64
 11  ORB     7 non-null      float64
 12  DRB     7 non-null      float64
 13  RPG     7 non-null      float64
 14  APG     7 non-null      float64
 15  SPG     7 non-null      float64
 16  BPG     7 non-null      float64
 17  TOV     7 non-null      float64
 18  TS%     7 non-null      float64
 19  eFG%    7 non-null      float64
 20  ORB%    7 non-null      float64
 21  DRB%    7 non-null      float64
 22  TRB%  

In [22]:
# Trazendo estatísticas dos dados com Describe
df_1lug.describe()

,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,...,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
count,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,...,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000,7.000000
mean,86.100000,30.285714,65.542857,0.462429,10.857143,30.185714,0.360143,14.657143,19.214286,0.760857,...,14.228571,8.757143,6.600000,116.857143,100.828571,16.028571,73.442857,1.528571,37.000000,0.855571
std,3.414674,0.864925,1.828348,0.019915,1.542571,3.936762,0.017014,1.275222,1.105183,0.032262,...,0.893895,1.056499,1.587451,3.392078,3.682714,2.886009,0.741299,0.149603,4.242641,0.049980
min,81.100000,29.100000,62.500000,0.435000,9.100000,26.600000,0.338000,12.800000,18.200000,0.694000,...,13.000000,7.200000,4.400000,111.600000,97.200000,13.500000,72.400000,1.300000,31.000000,0.800000
25%,83.950000,29.800000,64.800000,0.451000,9.750000,27.350000,0.350000,14.100000,18.600000,0.759500,...,13.700000,8.000000,5.650000,114.600000,97.800000,14.600000,72.950000,1.500000,34.000000,0.823000
50%,85.700000,29.900000,65.400000,0.453000,10.700000,28.900000,0.359000,14.500000,19.000000,0.764000,...,14.200000,9.100000,7.100000,117.600000,99.800000,15.200000,73.300000,1.500000,38.000000,0.840000
75%,88.600000,31.000000,66.600000,0.478500,11.700000,32.000000,0.369500,15.150000,19.300000,0.776500,...,14.650000,9.550000,7.450000,119.200000,103.100000,16.100000,74.050000,1.550000,40.000000,0.878000
max,90.800000,31.400000,68.100000,0.490000,13.300000,37.100000,0.385000,16.800000,21.500000,0.796000,...,15.700000,9.900000,8.500000,121.200000,107.000000,22.100000,74.400000,1.800000,42.000000,0.947000


O percentual de vitórias em uma temporada apresentou média de 85,5% e desvio padrão de 4.9%, com máximo de 94,7%.

# Preparação do modelo de machine learning
## Padronização de variáveis

In [23]:
'''
Como vou usar outer CV para que os hiperparâmetros sejam ajustados para cada temporada, vou criar uma coluna em cada dataframe de treino.
'''
df_17['Season'] = 2017
df_18['Season'] = 2018
df_20['Season'] = 2020
df_21['Season'] = 2021
df_22['Season'] = 2022
df_23['Season'] = 2023
df_24['Season'] = 2024
'''
Como o df_16 se apresentou como um outlier a ser tratado e o df_25 é o alvo para as previsões, um dataframe com as outras temporadas será criado
'''
df_modelo = pd.concat([df_17, df_18, df_20, df_21, df_22, df_23, df_24], ignore_index=True)
df_modelo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 35 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Team    116 non-null    object 
 1   PPG     116 non-null    float64
 2   FGM     116 non-null    float64
 3   FGA     116 non-null    float64
 4   FG%     116 non-null    float64
 5   3PM     116 non-null    float64
 6   3PA     116 non-null    float64
 7   3P%     116 non-null    float64
 8   FTM     116 non-null    float64
 9   FTA     116 non-null    float64
 10  FT%     116 non-null    float64
 11  ORB     116 non-null    float64
 12  DRB     116 non-null    float64
 13  RPG     116 non-null    float64
 14  APG     116 non-null    float64
 15  SPG     116 non-null    float64
 16  BPG     116 non-null    float64
 17  TOV     116 non-null    float64
 18  TS%     116 non-null    float64
 19  eFG%    116 non-null    float64
 20  ORB%    116 non-null    float64
 21  DRB%    116 non-null    float64
 22  TR

In [24]:
# Identificando colunas a serem padronizadas
pd.set_option('display.max_columns', None)
df_modelo.describe()

,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,ORB,DRB,RPG,APG,SPG,BPG,TOV,TS%,eFG%,ORB%,DRB%,TRB%,AST%,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %,Season
count,116.000000,116.000000,116.000000,116.000000,116.000000,116.00000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000,116.000000
mean,78.425000,27.945690,63.730172,0.438672,9.150000,27.24569,0.334802,13.381034,18.490517,0.723259,10.441379,26.086207,36.548276,16.793966,6.864655,2.037931,13.223276,0.545750,0.510388,28.520690,71.756897,50.095690,60.065517,15.533621,9.392241,5.615517,106.779310,107.546552,-0.768103,73.059483,1.281034,18.094828,0.481379,2020.922414
std,4.759483,1.673756,2.231405,0.024226,1.370592,2.92353,0.023780,1.508637,1.748091,0.037085,1.472535,1.875188,2.828688,1.915715,0.790295,0.478704,1.102437,0.031165,0.031031,3.183422,2.318104,2.584448,5.577347,1.182844,1.089966,1.251337,6.646261,4.722388,9.479182,1.644859,0.196895,9.358538,0.201105,2.352245
min,66.900000,23.600000,59.100000,0.381000,5.800000,20.00000,0.280000,10.300000,13.900000,0.617000,7.100000,21.100000,28.300000,12.500000,5.200000,1.000000,10.800000,0.483000,0.446000,20.300000,66.900000,42.600000,48.200000,12.900000,6.900000,2.900000,91.800000,96.200000,-20.700000,69.700000,0.900000,3.000000,0.107000,2017.000000
25%,75.125000,26.800000,62.200000,0.419750,8.175000,25.20000,0.318750,12.175000,17.300000,0.696500,9.650000,24.875000,34.500000,15.200000,6.275000,1.700000,12.575000,0.522750,0.486750,26.325000,69.975000,48.250000,56.475000,14.700000,8.600000,4.700000,101.775000,105.150000,-7.700000,71.900000,1.100000,10.000000,0.313750,2019.500000
50%,77.700000,27.700000,63.800000,0.435000,9.000000,27.05000,0.333000,13.450000,18.500000,0.723500,10.400000,26.100000,36.500000,16.800000,6.900000,2.000000,13.250000,0.542000,0.506500,28.900000,71.650000,50.050000,60.050000,15.500000,9.500000,5.450000,105.950000,107.850000,-1.200000,72.800000,1.300000,17.000000,0.471500,2021.000000
75%,81.300000,29.300000,65.200000,0.455250,10.000000,28.90000,0.351000,14.500000,19.750000,0.749250,11.200000,27.200000,38.725000,18.325000,7.500000,2.300000,13.800000,0.569000,0.535250,30.625000,73.500000,52.025000,63.875000,16.300000,10.200000,6.425000,111.600000,109.825000,5.750000,74.225000,1.400000,24.000000,0.626000,2023.000000
max,90.800000,31.700000,69.800000,0.490000,13.300000,37.10000,0.398000,16.900000,22.700000,0.800000,14.100000,30.600000,42.800000,20.700000,8.700000,3.600000,17.000000,0.617000,0.577000,35.500000,77.000000,56.000000,73.900000,19.300000,12.100000,8.600000,121.200000,119.600000,22.100000,77.400000,1.800000,42.000000,0.947000,2024.000000


## Ajuste de Hiperparâmetros com pipeline

In [25]:
'''
As colunas Teams e W's não serão usadas no modelo.
Vou utilizar o Standard Scaler e PCA no pipeline com outer CV para cada temporada. Essas medidas ajudam a definição melhor de hiperparâmetros para o modelo
'''
X_train = df_modelo.drop(columns=["Team", "W's", "Win %", "Season"])
Y_train = df_modelo["Win %"]
X_test = df_25.drop(columns=["Team", "W's", "Win %"])
Y_test = df_25["Win %"]
season_array = df_modelo["Season"].values

In [26]:
# Lista de hiperparâmetros para teste do Random Forest
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('model', RandomForestRegressor(random_state=42)),
])
param_rf = {
    'pca__n_components': [5, 6, 7, 8, 9],
    'model__n_estimators': [50, 100, 200, 300, 400, 500],
    'model__criterion': ['squared_error', 'absolute_error'],
    'model__max_features': ['sqrt', 'log2'],
    'model__max_depth': range(1, 50),
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

inner_search_rf = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_rf,
    n_iter=80,
    cv = 5,
    scoring='neg_mean_squared_error',
    n_jobs = -1,
    refit=True
)

# Lista para Elastic Net
pipe_en = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('model', ElasticNet(max_iter=5000))
])
param_en = {
    'pca__n_components': [5, 6, 7, 8, 9],
    'model__alpha': np.logspace(-4, 1, 30),
    'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}
inner_search_en = RandomizedSearchCV(
    estimator=pipe_en,
    param_distributions=param_en,
    n_iter=60,
    scoring='neg_mean_squared_error',
    cv=4,
    n_jobs=-1,
    random_state=42,
    refit=True
)

# Lista para XGBoost
pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('model', XGBRegressor(objective='reg:squarederror', verbosity=0))
])

param_xgb = {
    'pca__n_components': [5, 6, 7, 8, 9],
    'model__n_estimators': [50, 100, 200, 400],
    'model__max_depth': [2, 3, 4, 6, 8],
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2],
    'model__colsample_bytree': [0.5, 0.7, 1.0]
}
inner_search_xgb = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_xgb,
    n_iter=80,
    scoring='neg_mean_squared_error',
    cv=4,
    n_jobs=-1,
    random_state=42,
    refit=True
)

# Validação temporal por Season
outer_cv = GroupKFold(n_splits=7)

# Scores dos modelos
scores_rf = cross_val_score(
    inner_search_rf,
    X_train,
    Y_train,
    groups=season_array,
    cv=outer_cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1
)

scores_en = cross_val_score(
    inner_search_en,
    X_train,
    Y_train,
    groups=season_array,
    cv=outer_cv,
    scoring='neg_mean_squared_error',
    n_jobs=-1)

scores_xgb = cross_val_score(inner_search_xgb,
                             X_train,
                             Y_train,
                             groups=season_array,
                             cv=outer_cv,
                             scoring='neg_mean_squared_error',
                             n_jobs=-1)

print("Random Forest outer CV MSE (mean, std):", -scores_rf.mean(), scores_rf.std())
print("ElasticNet outer CV MSE (mean, std):", -scores_en.mean(), scores_en.std())
print("XGBoost outer CV MSE (mean, std):", -scores_xgb.mean(), scores_xgb.std())

Random Forest outer CV MSE (mean, std): 0.01006561215285458 0.004276007996089151
ElasticNet outer CV MSE (mean, std): 0.005942344467006976 0.002554010216966719
XGBoost outer CV MSE (mean, std): 0.009774775816368144 0.005087190568567105


Após análise dos resultados de erros quadrados médios dos hiperparâmetros pode-se observar:

* O modelo Elastic Net apresenta um erro médio nas previsões de 7.7 pontos percentuais, enquanto o XGboost e o Random Forest apresentam um erro médio de 9.8 pontos percentuais. Isso é um ganho de 38% para o primeiro modelo;
* O modelo será treinado de novo com o cross-validation dentro do teste de hiperparâmetros para uma estimativa melhor de generalização antes do teste

## Treino 2 do modelo e definição dos melhores hiperparâmetros

In [27]:
inner_search_rf_full = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=param_rf,
    n_iter=80,
    cv = GroupKFold(n_splits=7), # Cross validation agrupado por season
    scoring='neg_mean_squared_error',
    n_jobs = -1,
    random_state=42,
    refit=True
)

inner_search_en_full = RandomizedSearchCV(
    estimator=pipe_en,
    param_distributions=param_en,
    n_iter=60,
    scoring='neg_mean_squared_error',
    cv=GroupKFold(n_splits=7),
    n_jobs=-1,
    random_state=42,
    refit=True
)

inner_search_xgb_full = RandomizedSearchCV(
    estimator=pipe_xgb,
    param_distributions=param_xgb,
    n_iter=80,
    scoring='neg_mean_squared_error',
    cv=GroupKFold(n_splits=7),
    n_jobs=-1,
    random_state=42,
    refit=True
)

In [28]:
# Modelagem com o treino refeito
inner_search_en_full.fit(X_train, Y_train, groups=season_array)
inner_search_rf_full.fit(X_train, Y_train, groups=season_array)
inner_search_xgb_full.fit(X_train, Y_train, groups=season_array)

# Melhores estimadores
best_en = inner_search_en_full.best_estimator_
best_rf = inner_search_rf_full.best_estimator_
best_xgb = inner_search_xgb_full.best_estimator_

# Previsões dos modelos para a season teste
en_pred = best_en.predict(X_test)
rf_pred = best_rf.predict(X_test)
xgb_pred = best_xgb.predict(X_test)

## Resultados dos testes e avaliação

In [29]:
# Scores de R2
r2_en = r2_score(Y_test, en_pred)
r2_rf = r2_score(Y_test, rf_pred)
r2_xgb = r2_score(Y_test, xgb_pred)

# Scores de mean squared error
mse_en = mean_squared_error(Y_test, en_pred)
mse_rf = mean_squared_error(Y_test, rf_pred)
mse_xgb = mean_squared_error(Y_test, xgb_pred)

# Resultados
print(f"Resultados do Elastic Net. R2: {r2_en}. MSE: {mse_en}")
print(f"Resultados do Random Forest. R2: {r2_rf}. MSE: {mse_rf}")
print(f"Resultados do XGboost. R2: {r2_xgb}. MSE: {mse_xgb}")

Resultados do Elastic Net. R2: 0.9559351744750954. MSE: 0.002182975862986331
Resultados do Random Forest. R2: 0.8703204306713141. MSE: 0.006424338832500031
Resultados do XGboost. R2: 0.9152803988893227. MSE: 0.0041970175109830666


Os resultados obtidos indicam que o modelo linear regularizado (Elastic Net) explica melhor a variação de Win%.

O R2 alto e MSE baixo indicam previsões mais estáveis entre seasons, podendo ser valioso para metas operacionais.

## Gráfico que demonstra visualmente a performance do teste do modelo de Elastic Net

In [30]:
# Montando dataframe para gerar o gráfico
df_en = pd.DataFrame({
    'y_true': np.asarray(Y_test).ravel(),
    'y_pred': np.asarray(en_pred).ravel()
}, index=X_test.index)

df_en = df_en.sort_values('y_true').reset_index(drop=True)

# Gerando Gráfico
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_en.index,
    y=df_en['y_true'],
    mode='markers',
    name='Y_test (real)',
    marker = dict(color='#A1A2A5', size=6)
))
fig.add_trace(go.Scatter(
    x=df_en.index,
    y=df_en['y_pred'],
    mode='lines',
    name='Previsão Elastic Net',
    line=dict(color='#AB9034', width=2)
))
fig.update_layout(
    template = "simple_white",
    xaxis=dict(
        title=dict(
            text="Amostra (ordenada por Y_test)",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    yaxis=dict(
        title=dict(
            text = "Porcentagem de vitórias",
            font=dict(family="Segoe UI", color="black")),
        tickfont=dict(family="Segoe UI", color="black"),
        linecolor="black",
    ),
    title =dict(
        text=f"<b>R2 score de Elastic Net. R²: {r2_en:.3f} <b>",
        font=dict(family="Segoe UI", size=20, color="black")
    )
)
fig.show()

In [40]:
'''
Montando um stack com os modelos testados para tentar obter um algoritmo com resultados mais robustos.
Como o Pipeline e o RandomSearch já foram feitos, só será necessário realizar o stacking. O estimador final será o ElasticNet pois ele obteve o melhor resultado no outros treinos.
'''

# Montando o stack
en_stack = StackingRegressor(
    estimators=[('rf', best_rf), ('xgb', best_xgb)],
    final_estimator=ElasticNet(max_iter=5000),
    cv = 5,
    passthrough=True,
    n_jobs=-1,
    verbose=1
)

# Treinamento do Stack
en_stack.fit(X_train, Y_train)

,"estimators estimators: list of (str, estimator)Base estimators which will be stacked together. Each element of thelist is defined as a tuple of string (i.e. name) and an estimatorinstance. An estimator can be set to 'drop' using `set_params`.","[('rf', ...), ('xgb', ...)]"
,"final_estimator final_estimator: estimator, default=NoneA regressor which will be used to combine the base estimators.The default regressor is a :class:`~sklearn.linear_model.RidgeCV`.",ElasticNet(max_iter=5000)
,"cv cv: int, cross-validation generator, iterable, or ""prefit"", default=NoneDetermines the cross-validation splitting strategy used in`cross_val_predict` to train `final_estimator`. Possible inputs forcv are:* None, to use the default 5-fold cross validation,* integer, to specify the number of folds in a (Stratified) KFold,* An object to be used as a cross-validation generator,* An iterable yielding train, test splits,* `""prefit""`, to assume the `estimators` are prefit. In this case, the estimators will not be refitted.For integer/None inputs, if the estimator is a classifier and y iseither binary or multiclass,:class:`~sklearn.model_selection.StratifiedKFold` is used.In all other cases, :class:`~sklearn.model_selection.KFold` is used.These splitters are instantiated with `shuffle=False` so the splitswill be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.If ""prefit"" is passed, it is assumed that all `estimators` havebeen fitted already. The `final_estimator_` is trained on the `estimators`predictions on the full training set and are **not** cross validatedpredictions. Please note that if the models have been trained on the samedata to train the stacking model, there is a very high risk of overfitting... versionadded:: 1.1 The 'prefit' option was added in 1.1.. note:: A larger number of split will provide no benefits if the number of training samples is large enough. Indeed, the training time will increase. ``cv`` is not used for model evaluation but for prediction.",5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for `fit` of all `estimators`.`None` means 1 unless in a `joblib.parallel_backend` context. -1 meansusing all processors. See :term:`Glossary ` for more details.",-1
,"passthrough passthrough: bool, default=FalseWhen False, only the predictions of estimators will be used astraining data for `final_estimator`. When True, the`final_estimator` is trained on the predictions as well as theoriginal training data.",True
,"verbose verbose: int, default=0Verbosity level.",1
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case result

In [41]:
stack_pred = en_stack.predict(X_test)

r2_en_stack = r2_score(Y_test, stack_pred)
mse_en_stack = mean_squared_error(Y_test, stack_pred)

print(f"Resultados do Elastic Net Stack. R2: {r2_en_stack}. MSE: {mse_en_stack}")

Resultados do Elastic Net Stack. R2: 0.8817855597249803. MSE: 0.005856355192668505


O stack dos modelos apresentou um desempenho inferior ao ElasticNet, por isso este foi escolhido.

Como o modelo de Elastic Net foi escolhido, agora é necessário projetar as variáveis de volta para saber o efeito linear de cada estatística sobre o Win%.

Depois as mudanças serão aplicadas e uma nova previsão será feita.

## Tradução em metas operacionais

In [31]:
'''
Reversão do Scaler e obtenção das variáveis usadas no PCA com Intercept ajustado
'''
def get_scaler_pipeline(pipeline):
    if 'scaler' in pipeline.named_steps:
        return pipeline.named_steps['scaler'], None
    raise ValueError("Não foi possível localizar um StandardScaler no Pipeline")

def elastic_to_original(pipeline, X_reference):
    # Extrai componentes
    pca = pipeline.named_steps['pca']
    model = pipeline.named_steps['model']

    # Obtém scaler e colunas escaladas
    scaler, scaler_cols = get_scaler_pipeline(pipeline)

    # Nomes das features originais
    if hasattr(X_reference, 'columns'):
        feature_names = list(X_reference.columns)
    else:
        # fallback: cria nomes genéricos
        n_features = pca.components_.shape[1]
        feature_names = [f'X{i}' for i in range(n_features)]

    # Número de componentes usados no PCA
    n_comp = pca.n_components_
    # Matriz de componentes: shape (n_components, n_features)
    V = pca.components_[:n_comp, :]            # cada linha é um componente
    # Coeficientes do modelo no espaço dos PCs
    beta_pca = model.coef_.ravel()

    # Coeficientes no espaço escalado (antes de inverter o scaler)
    # Z = X_scaled @ V.T  => coef_scaled = V.T @ beta_pca
    coef_scaled = V.T.dot(beta_pca)            # shape (n_features,)

    # obter média e escala do scaler
    mu = scaler.mean_
    sigma = scaler.scale_

    # coeficientes na escala original
    coef_orig = coef_scaled / sigma

    # intercept ajustado para a escala original
    intercept_scaled = model.intercept_
    intercept_orig = intercept_scaled - np.sum(coef_orig * mu)

    # montar DataFrame de saída
    df_coef = pd.DataFrame({
        'feature': feature_names,
        'coef_scaled_space': coef_scaled,
        'scale': sigma,
        'coef_original_space': coef_orig
    })

    return df_coef.set_index('feature'), intercept_orig


coef_df, intercept_original = elastic_to_original(best_en, X_train)

# Exibir os 10 maiores em magnitude
coef_df['abs_coef'] = coef_df['coef_original_space'].abs()
coef_df_sorted = coef_df.sort_values('abs_coef', ascending=False)
print("Intercept (escala original):", intercept_original)
print(coef_df_sorted[['coef_original_space']].head(10))

Intercept (escala original): -0.8922491786696867
         coef_original_space
feature                     
FG%                 0.586553
3P%                 0.395115
TS%                 0.386241
eFG%                0.382010
FT%                 0.249253
Ast/TO              0.074449
BPG                 0.017217
TOV                -0.015072
SPG                 0.013408
STL%                0.012729


Os valores ao lado das estatísticas indicam o impacto percentual esperado nas estatísticas de Win% para variação de unidade (mantendo as outras constantes)

# O que mudar para que o Brasília alcance 85% de Win%?

## Dados do Brasília na temporada 2025/2026

In [40]:
# Criando uma linha só do Brásilia
df_bsb_25 = df_25[df_25['Team'] == "Uniceub-BRB-Brasilia"]
df_bsb_25

,Team,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,ORB,DRB,RPG,APG,SPG,BPG,TOV,TS%,eFG%,ORB%,DRB%,TRB%,AST%,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
8,Uniceub-BRB-Brasilia,81.6,28.1,65.1,0.432,10.4,32.1,0.324,14.9,20.1,0.742,11.6,28.1,39.7,19.0,8.4,2.4,13.4,0.551,0.512,30.4,71.5,51.2,67.4,15.4,11.2,7.2,109.2,96.3,12.9,73.8,1.4,28,0.737


In [41]:
'''
Com 32% médio de acertos de 3pts o time do Brasília conseguiu obter 73% de vitórias na temporada.
O Objetivo é chegar aos 85% de vitória.
Vamos ver o quanto que 6% a mais de acertos de 3pts em média consegue produzir.
Todas as estatísticas relacionadas serão ajustadas proporcionalmente.
'''
df_bsb_27 = df_bsb_25.assign(**{
    '3P%' : df_bsb_25['3P%'] + 0.056,
    'PPG' : df_bsb_25['PPG'] + 14.1,
    'FGM' : df_bsb_25['FGM'] + 4.85,
    'FG%' : df_bsb_25['FG%'] + 0.074,
    '3PM' : df_bsb_25['3PM'] + 1.79,
    'TS%' : df_bsb_25['TS%'] + 0.095,
    'eFG%' : df_bsb_25['eFG%'] + 0.088,
    'ORtg' : df_bsb_25['ORtg'] + 18.87,
    'eDiff' : df_bsb_25['eDiff'] + 2.22,
})

df_bsb_27

,Team,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,ORB,DRB,RPG,APG,SPG,BPG,TOV,TS%,eFG%,ORB%,DRB%,TRB%,AST%,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
8,Uniceub-BRB-Brasilia,95.7,32.95,65.1,0.506,12.19,32.1,0.38,14.9,20.1,0.742,11.6,28.1,39.7,19.0,8.4,2.4,13.4,0.646,0.6,30.4,71.5,51.2,67.4,15.4,11.2,7.2,128.07,96.3,15.12,73.8,1.4,28,0.737


In [42]:
# Realizando previsões
bsb_27 = df_bsb_27.drop(columns=["Team", "W's", "Win %"])
pred_27 = best_en.predict(bsb_27)
print(f"Previsão de vitórias para a temporada 2026/2027 do Brasília: {pred_27[0]:.3f}")

Previsão de vitórias para a temporada 2026/2027 do Brasília: 0.933


Subindo o acerto percentual de 3pt de 32% para 38% obtivemos 93% de vitórias na temporada, com erro médio de 4 pontos percentuais. Como não é realista vou diminuir para 35% de acertos.

In [45]:
df_bsb_27 = df_bsb_25.assign(**{
    '3P%' : df_bsb_25['3P%'] + 0.036,
    'PPG' : df_bsb_25['PPG'] + 9.06,
    'FGM' : df_bsb_25['FGM'] + 3.12,
    'FG%' : df_bsb_25['FG%'] + 0.047,
    '3PM' : df_bsb_25['3PM'] + 1.15,
    'TS%' : df_bsb_25['TS%'] + 0.061,
    'eFG%' : df_bsb_25['eFG%'] + 0.056,
    'ORtg' : df_bsb_25['ORtg'] + 12.13,
    'eDiff' : df_bsb_25['eDiff'] + 1.43,
})

df_bsb_27

,Team,PPG,FGM,FGA,FG%,3PM,3PA,3P%,FTM,FTA,FT%,ORB,DRB,RPG,APG,SPG,BPG,TOV,TS%,eFG%,ORB%,DRB%,TRB%,AST%,TOV%,STL%,BLK%,ORtg,DRtg,eDiff,Pace,Ast/TO,W's,Win %
8,Uniceub-BRB-Brasilia,90.66,31.22,65.1,0.479,11.55,32.1,0.36,14.9,20.1,0.742,11.6,28.1,39.7,19.0,8.4,2.4,13.4,0.612,0.568,30.4,71.5,51.2,67.4,15.4,11.2,7.2,121.33,96.3,14.33,73.8,1.4,28,0.737


In [46]:
bsb_27 = df_bsb_27.drop(columns=["Team", "W's", "Win %"])
pred_27 = best_en.predict(bsb_27)
print(f"Previsão de vitórias para a temporada 2026/2027 do Brasília: {pred_27[0]:.3f}")

Previsão de vitórias para a temporada 2026/2027 do Brasília: 0.843


Conforme o modelo, o time do Brasília poderia aumentar os acertos percentuais médios de 3pt em 4 pontos percentuais, ou cerca de 1 cesta de 3pts a mais por jogo, para obter 84% de vitórias totais na temporada regular.